# LC 131 — Palindrome Partitioning
**Difficulty:** Medium | **Pattern:** Backtracking

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Walk the string with a
<code>start</code> pointer. At each position, try every possible
<em>end</em> for the next substring. Only recurse if that substring
is a palindrome — otherwise skip it entirely. When
<code>start == len(s)</code> you have consumed the whole string,
so record the current path as a valid partition.
</div>

## Official Problem Statement

Given a string `s`, partition `s` such that every substring of the
partition is a **palindrome**. Return all possible palindrome
partitioning of `s`.

**Constraints:**
- `1 <= s.length <= 16`
- `s` contains only lowercase English letters.

## What This Is Actually Asking

Cut the string into pieces (substrings) in every possible way.
Keep only the cuttings where **every piece is a palindrome**.
Return all such valid cuttings.

For `"aab"` the cuts are:
- `["a","a","b"]` — all three are palindromes ✓
- `["aa","b"]` — `"aa"` and `"b"` are palindromes ✓
- `["aab"]` — `"aab"` is not a palindrome ✗
- `["a","ab"]` — `"ab"` is not a palindrome ✗

The key challenge: generate cuts efficiently by only extending
a path when the current piece is confirmed as a palindrome.

## Walk Through an Example by Hand

`s = "aab"`

`backtrack(start=0, path=[])`:

- Try `end=1`: `s[0:1]="a"` → palindrome?
  - `is_palindrome(s,0,0)` → yes (single char)
  - `path=["a"]`, `backtrack(1, ["a"])`:
    - Try `end=2`: `s[1:2]="a"` → palindrome? yes
      - `path=["a","a"]`, `backtrack(2, ["a","a"])`:
        - Try `end=3`: `s[2:3]="b"` → palindrome? yes
          - `path=["a","a","b"]`, `backtrack(3, ...):`
            - `start==len(s)=3` → **record** `["a","a","b"]`
    - Try `end=3`: `s[1:3]="ab"` → palindrome? no → skip
- Try `end=2`: `s[0:2]="aa"` → palindrome?
  - `is_palindrome(s,0,1)` → yes
  - `path=["aa"]`, `backtrack(2, ["aa"])`:
    - Try `end=3`: `s[2:3]="b"` → palindrome? yes
      - `path=["aa","b"]`, `backtrack(3, ...)`:
        - `start==3` → **record** `["aa","b"]`
- Try `end=3`: `s[0:3]="aab"` → palindrome? no → skip

Result: `[["a","a","b"], ["aa","b"]]`

## The Picture

Decision tree for `s = "aab"`:

```
                  start=0, path=[]
           /           |           \
   "a"(pal)       "aa"(pal)     "aab"(not pal)
  start=1           start=2          PRUNE
   /    \
"a"(pal) "ab"(not)
start=2   PRUNE
  |
"b"(pal)
start=3
  |              |
RECORD           "b"(pal)
[a,a,b]         start=3
                  |
                RECORD
                [aa,b]

is_palindrome(s, l, r): two pointers
  while l < r: if s[l] != s[r]: return False
               l += 1; r -= 1
  return True
```

**Key rules:**
- Only branch when `s[start:end]` is a palindrome
- Base case: `start == len(s)` means full string consumed
- `end` ranges from `start+1` to `len(s)+1` (inclusive)

## When To Use This Pattern

Use **Backtracking with Validity Check Before Recurse** when:

| Signal | Example |
|--------|---------|
| Partition a sequence into valid segments | This problem |
| Enumerate all valid decompositions | Word Break II |
| Check a constraint before going deeper | Any cut problem |
| Input size small (n ≤ 16) | Constraint signals backtracking |

**Contrast with:**
- LC 132 (Palindrome Partitioning II) — min cuts, use DP
- LC 139 (Word Break) — count/boolean, use DP
- LC 140 (Word Break II) — enumerate, similar backtracking

## The Approach

**Algorithm — Backtracking with palindrome check:**

```
def is_palindrome(s, l, r):
    while l < r:
        if s[l] != s[r]: return False
        l += 1; r -= 1
    return True

backtrack(start, path):
    if start == len(s):     # consumed entire string
        result.append(path.copy())
        return
    for end in range(start + 1, len(s) + 1):
        if is_palindrome(s, start, end - 1):
            path.append(s[start:end])
            backtrack(end, path)   # move start forward to end
            path.pop()             # backtrack
```

**Why `end - 1` in `is_palindrome`?**
Python slice `s[start:end]` is exclusive on the right, but our
two-pointer check uses inclusive indices. So we pass `end - 1`
as the right boundary to the helper.

In [ ]:
from typing import List

In [ ]:
def test_harness(func):
    """
    Run test cases for partition.
    Order-independent: each partition is sorted internally,
    then the list of partitions is sorted for comparison.
    """
    def norm(parts):
        return sorted([list(p) for p in parts])

    cases = [
        # (s, expected)
        ("aab",  [["a", "a", "b"], ["aa", "b"]]),
        ("a",    [["a"]]),
        ("ab",   [["a", "b"]]),
        ("aa",   [["a", "a"], ["aa"]]),
        ("aba",  [["a", "b", "a"], ["aba"]]),
    ]

    passed = 0
    for i, (s, expected) in enumerate(cases, 1):
        result = func(s)
        ok = norm(result) == norm(expected)
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        else:
            print(f"  Case {i} {status}")
            print(f"    s={s!r}")
            print(f"    expected: {norm(expected)}")
            print(f"    got:      {norm(result)}")

    total = len(cases)
    print(f"\nResult: {passed}/{total} passed",
          "✓" if passed == total else "✗")

In [ ]:
def partition(s: str) -> List[List[str]]:
    """
    Return all ways to partition s such that every substring
    in the partition is a palindrome.

    Approach:
        Backtrack from index start. At each step, try every
        possible next segment s[start:end]. Only recurse if
        that segment is a palindrome (checked via two pointers
        on indices start..end-1). Base case: start==len(s)
        means the whole string is consumed — record the path.

    Args:
        s: input string (1 <= len(s) <= 16)

    Returns:
        list of partitions; each partition is a list of
        palindromic substrings that concatenate to s

    Examples:
        >>> partition("aab")
        [["a","a","b"], ["aa","b"]]
        >>> partition("a")
        [["a"]]
    """
    pass

    # Debug hints (uncomment to trace execution):
    # print(f"backtrack(start={start}, path={path})")
    # print(f"  trying segment {s[start:end]!r},"
    #       f" palindrome={is_palindrome(s, start, end-1)}")

In [ ]:
# Uncomment and run when solution is ready
# test_harness(partition)

## Complexity

| | Value | Reason |
|-|-------|--------|
| **Time** | O(N * 2^N) | 2^N possible cuts; O(N) to check/copy each |
| **Space** | O(N) | Recursion depth and path both at most N deep |

**Optimization note:** For repeated calls (not needed here at
n ≤ 16), precompute a 2D `dp[i][j]` palindrome table in O(N²)
so each `is_palindrome` check is O(1) lookup instead of O(N).

## Real World Connection

**Natural language tokenization** — given a raw character stream,
find all valid ways to segment it into known tokens (words,
morphemes). The palindrome check here is analogous to a
dictionary lookup: only extend the path if the segment is valid.

**DNA sequence analysis** — bioinformaticians partition genetic
sequences into restriction enzyme recognition sites (which often
have palindromic structure in the biological sense). Enumerating
all valid cut points uses exactly this backtracking pattern where
each segment must satisfy a structural property before the search
continues deeper.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra